<a href="https://colab.research.google.com/github/gyan1131/Quantum_Computing_2026/blob/main/minimal%20reproduction%20script%20for%20GitHub%20issue%20reporting_AerSimulator.save_expectation_value_%232442.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install qiskit qiskit-aer

In [10]:
import qiskit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Pauli  # Use Pauli for observables

# Create a quantum circuit with 2 qubits
circ = qiskit.QuantumCircuit(2, 2)

# Apply a Hadamard gate to qubit 0
circ.h(0)

# Qubit 1 remains idle

# Define the observable for qubit 1 (the idle qubit)
# In modern Qiskit, you define a Pauli string directly for the observable.
# For Z on qubit 1, in a 2-qubit system, this is 'IZ'.
observable_q1_pauli = Pauli('IZ') # I on q0, Z on q1

# Configure AerSimulator
simulator = AerSimulator(method='statevector')

print(f"Original Circuit for first approach:\n{circ.draw(output='text')}")

# First approach: Using save_expectation_value with a Pauli observable
circ_measure_q1_pauli = circ.copy()
# save_expectation_value takes the observable and the qubits it acts on. It expects a Pauli or SparsePauliOp.
# For an observable like 'IZ', we can pass Pauli('IZ') and specify all qubits, or
# more simply, if we only care about Z on qubit 1, we can pass Pauli('Z') and specify only qubit 1.
# Given the bug report implies 'subsystem expectation', let's be explicit for a 2-qubit observable.
circ_measure_q1_pauli.save_expectation_value(observable_q1_pauli, [0, 1])

# Run the simulation, passing the modified circuit object
job_exp_val_pauli = simulator.run(circ_measure_q1_pauli, shots=1) # shots=1 for statevector sim
result_exp_val_pauli = job_exp_val_pauli.result()

exp_val_data_pauli = result_exp_val_pauli.data()
print(f"\nResult data keys from save_expectation_value (Pauli): {list(exp_val_data_pauli.keys())}")

# Correctly extract the expectation value
exp_val_pauli = exp_val_data_pauli.get('expectation_value', 'Not Found') # Use .get for robustness

print(f"Circuit with save_expectation_value for IZ on Qubits [0,1]:\n{circ_measure_q1_pauli.draw(output='text')}")
print(f"Observable for IZ: {observable_q1_pauli}")
print(f"Expectation value for IZ on idle qubit 1 (first approach): {exp_val_pauli}")


# Second approach (retained and updated): Using save_expectation_value with Pauli('Z') on qubit 1
circ2 = qiskit.QuantumCircuit(2, name='test_idle_qbit')
circ2.h(0)

# Try to get expectation value of Z on q1. This implicitly means 'IZ' for a 2-qubit system.
# `save_expectation_value` can take a Pauli operator and a list of qubits it applies to.
circ2.save_expectation_value(Pauli('Z'), [1]) # Modify circ2 in place
job2 = simulator.run(circ2, shots=1) # Pass the modified circ2
result2 = job2.result()

# The key for this will be 'expectation_value'
exp_val2 = result2.data()['expectation_value']

print(f"\nCircuit 2 (second approach, direct save_expectation_value with Pauli('Z') on [1]):\n{circ2.draw(output='text')}")
print(f"Expectation value for Z on idle qubit 1 using save_expectation_value: {exp_val2}")

# The bug report suggests this value might be wrong when other qubits are active or it's a subsystem.


Original Circuit for first approach:
     ┌───┐
q_0: ┤ H ├
     └───┘
q_1: ─────
          
c: 2/═════
          

Result data keys from save_expectation_value (Pauli): ['counts', 'expectation_value']
Circuit with save_expectation_value for IZ on Qubits [0,1]:
     ┌───┐ expectation_valu... 
q_0: ┤ H ├──────────░──────────
     └───┘          ░          
q_1: ───────────────░──────────
                    ░          
c: 2/══════════════════════════
                               
Observable for IZ: IZ
Expectation value for IZ on idle qubit 1 (first approach): 2.220446049250313e-16

Circuit 2 (second approach, direct save_expectation_value with Pauli('Z') on [1]):
             ┌───┐        
q_0: ────────┤ H ├────────
             └───┘        
      expectation_valu... 
q_1: ──────────░──────────
               ░          
Expectation value for Z on idle qubit 1 using save_expectation_value: 1.0


## Minimal Reproduction Script for GitHub Issue

In [11]:
import qiskit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Pauli

# --- Setup the circuit ---
# A 2-qubit circuit where qubit 0 is in superposition and qubit 1 is idle (in |0> state)
circ = qiskit.QuantumCircuit(2)
circ.h(0)

simulator = AerSimulator(method='statevector')

print(f"Initial Circuit:\n{circ.draw(output='text')}")

# --- Case 1: Incorrect Expectation Value (Bug Reproduction) ---
# Try to measure the expectation value of 'IZ' on both qubits [0,1].
# This means Identity on qubit 0 and Z on qubit 1.
# Expected value for q1 (idle, in |0>) with Z is +1.
# Expected value for q0 (H|0>) with I is +1.
# So, the overall expectation for IZ should be +1.

circ_buggy = circ.copy()
observable_iz = Pauli('IZ') # I on q0, Z on q1
circ_buggy.save_expectation_value(observable_iz, [0, 1])

job_buggy = simulator.run(circ_buggy, shots=1)
result_buggy = job_buggy.result()

exp_val_buggy = result_buggy.data().get('expectation_value')

print("\n--- First Approach (Buggy Case: Pauli('IZ') on [0,1]) ---")
print(f"Circuit used:\n{circ_buggy.draw(output='text')}")
print(f"Observable: {observable_iz}")
print(f"Resulting expectation value: {exp_val_buggy}")
print("Expected: 1.0 (I on H|0> is 1, Z on |0> is 1, product is 1)")
print("This value is INCORRECT and reproduces the bug.")

# --- Case 2: Correct Expectation Value ---
# Try to measure the expectation value of 'Z' specifically on qubit 1.
# The simulator should correctly deduce this as 'IZ' effectively and calculate.
# Expected value for q1 (idle, in |0>) with Z is +1.

circ_correct = circ.copy()
observable_z_q1 = Pauli('Z')
circ_correct.save_expectation_value(observable_z_q1, [1]) # Acting only on qubit 1

job_correct = simulator.run(circ_correct, shots=1)
result_correct = job_correct.result()

exp_val_correct = result_correct.data().get('expectation_value')

print("\n--- Second Approach (Correct Case: Pauli('Z') on [1]) ---")
print(f"Circuit used:\n{circ_correct.draw(output='text')}")
print(f"Observable: {observable_z_q1} on qubit 1")
print(f"Resulting expectation value: {exp_val_correct}")
print("Expected: 1.0 (Z on |0> is 1)")
print("This value is CORRECT.")


Initial Circuit:
     ┌───┐
q_0: ┤ H ├
     └───┘
q_1: ─────
          

--- First Approach (Buggy Case: Pauli('IZ') on [0,1]) ---
Circuit used:
     ┌───┐ expectation_valu... 
q_0: ┤ H ├──────────░──────────
     └───┘          ░          
q_1: ───────────────░──────────
                    ░          
Observable: IZ
Resulting expectation value: 2.220446049250313e-16
Expected: 1.0 (I on H|0> is 1, Z on |0> is 1, product is 1)
This value is INCORRECT and reproduces the bug.

--- Second Approach (Correct Case: Pauli('Z') on [1]) ---
Circuit used:
             ┌───┐        
q_0: ────────┤ H ├────────
             └───┘        
      expectation_valu... 
q_1: ──────────░──────────
               ░          
Observable: Z on qubit 1
Resulting expectation value: 1.0
Expected: 1.0 (Z on |0> is 1)
This value is CORRECT.
